# Compare object detectors from data to INT8

This notebook runs the same workflow as the command-line script. Start with COCO128, then point the configuration at a labeled insect dataset. Set up the Python environment described in [the workflow guide](detection/README.md) before running the cells.

In [ ]:
from datetime import datetime, timezone
import json
from pathlib import Path
import subprocess
import sys

notebooks = Path.cwd()
if (notebooks / 'Notebooks' / 'detection').is_dir():
    notebooks /= 'Notebooks'
pipeline = notebooks / 'detection' / 'pipeline.py'
assert pipeline.is_file(), 'Open Jupyter from the repository root or Notebooks directory.'

models = ['yolov8n', 'yolo11n', 'mobilenetv3-large']
dataset = None  # Path('/path/to/insect_dataset')
classes = None  # Path('/path/to/classes.json')
epochs = 1
size = 256
batch = 8
device = 'cpu'
seed = 42
stamp = datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')
runs = notebooks.parent / 'detection-runs'

def command(output):
    args = [sys.executable, str(pipeline), '--out', str(output),
            '--models', *models, '--epochs', str(epochs), '--size', str(size),
            '--batch', str(batch), '--device', device, '--seed', str(seed)]
    if dataset is not None:
        args += ['--dataset', str(dataset), '--classes', str(classes)]
    return args

## Inspect the dataset

Training updates weights, validation selects the checkpoint, and the held-out test split measures the final exports. INT8 calibration uses training images. Check the preview for each split before continuing.

In [ ]:
from IPython.display import Image, display

inspection = runs / f'{stamp}-inspection'
subprocess.run([*command(inspection), '--prepare-only'], check=True)
settings = json.loads((inspection / 'run.json').read_text())
print(settings['split_counts'])
for split in ('train', 'val', 'test'):
    print(split)
    display(Image(filename=str(inspection / f'preview-{split}.jpg')))

## Train and export

The one-epoch default exercises the pipeline. Each detector gets its own training curves and checkpoint. Both exports use the same test images, input size and CPU inference batch size.

In [ ]:
output = runs / stamp
args = command(output)
if dataset is None:
    inspected_data = inspection / 'coco128-data'
    class_file = inspection / 'classes.json'
    class_file.write_text(json.dumps(settings['arguments']['classes']))
    args += ['--dataset', str(inspected_data), '--classes', str(class_file)]
subprocess.run(args, check=True)

## Compare accuracy and inference time

Use mAP50–95 for the main detection comparison. Inference time here measures the model call on this host, excluding preprocessing and NMS. Each INT8 export has a JSON sidecar with its class order and tensor interface.

In [ ]:
import pandas as pd

report = json.loads((output / 'comparison.json').read_text())
rows = [{'model': model, 'precision': precision, **results[precision]}
        for model, results in report.items() for precision in ('FP32', 'INT8')]
display(pd.DataFrame(rows).set_index(['model', 'precision']))
for model in models:
    curve = output / model / 'training' / 'results.png'
    if curve.is_file():
        print(model)
        display(Image(filename=str(curve)))